<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L8/kmeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K-Means Clustering — From Scratch

1. **Implement K-Means from scratch** using only NumPy
2. **Animate** the algorithm step by step
3. **Elbow Method** to choose K
4. **Compare** with scikit-learn

---

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.animation as animation
from sklearn.datasets import make_blobs

K = 3
X, y = make_blobs(n_samples=240, centers=K, cluster_std=3.0, random_state=42)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=20, alpha=0.6, color='gray')
plt.title('Our Dataset (unlabelled)', fontsize=14)
plt.xlabel('x₁'); plt.ylabel('x₂')
plt.tight_layout()
plt.show()
print(f'Samples: {X.shape[0]}  |  Features: {X.shape[1]}')

---
## 1. K-Means from Scratch

Four helper functions + one wrapper that records every step for animation.

In [ ]:
def initialize_centroids(X, k):
    """K-Means++ initialisation: pick centroids that are spread apart."""
    n = X.shape[0]
    centroids = [X[np.random.randint(n)]]
    for _ in range(1, k):
        dists = np.min([np.sum((X - c) ** 2, axis=1) for c in centroids], axis=0)
        centroids.append(X[np.random.choice(n, p=dists / dists.sum())])
    return np.array(centroids)


def compute_distances(X, centroids):
    """Euclidean distance from every point to every centroid → (n_samples, k)."""
    return np.sqrt(((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2).sum(axis=2))


def assign_clusters(distances):
    """Assign each sample to the nearest centroid."""
    return np.argmin(distances, axis=1)


def update_centroids(X, labels, k):
    """Recompute each centroid as the mean of its assigned points."""
    new_centroids = np.zeros((k, X.shape[1]))
    for i in range(k):
        members = X[labels == i]
        if len(members) > 0:
            new_centroids[i] = members.mean(axis=0)
    return new_centroids


def compute_wcss(X, labels, centroids):
    """Within-Cluster Sum of Squares (inertia)."""
    return sum(((X[labels == i] - centroids[i]) ** 2).sum()
               for i in range(len(centroids)))


def kmeans(X, k, max_iters=100, seed=0):
    """Full K-Means. Returns centroids, labels, wcss, and history."""
    np.random.seed(seed)
    centroids = initialize_centroids(X, k)
    history = {'centroids': [centroids.copy()], 'labels': []}

    for _ in range(max_iters):
        labels = assign_clusters(compute_distances(X, centroids))
        history['labels'].append(labels.copy())

        new_centroids = update_centroids(X, labels, k)
        history['centroids'].append(new_centroids.copy())

        if np.linalg.norm(new_centroids - centroids) < 1e-6:
            break
        centroids = new_centroids

    return centroids, labels, compute_wcss(X, labels, centroids), history


centroids, labels, wcss, history = kmeans(X, K)
print(f'Converged in {len(history["labels"])} iterations  |  WCSS = {wcss:.2f}')

---
## 2. 🎬 Animate the Algorithm

Each frame alternates between **assigning points** and **moving centroids** (arrows show movement).

In [ ]:
COLORS = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6', '#1abc9c']
hist = history

fig, ax = plt.subplots(figsize=(6, 5))

def animate_frame(frame_idx):
    ax.clear()
    n_iters = len(hist['labels'])
    iter_idx = min(frame_idx // 2, n_iters - 1)
    is_update = frame_idx % 2 == 1

    labels_now = hist['labels'][iter_idx]
    c_before = hist['centroids'][iter_idx]
    c_after  = hist['centroids'][min(iter_idx + 1, len(hist['centroids']) - 1)]
    c_show   = c_after if is_update else c_before

    for i in range(K):
        mask = labels_now == i
        ax.scatter(X[mask, 0], X[mask, 1], s=20, alpha=0.5, color=COLORS[i])

    if is_update:
        for old, new in zip(c_before, c_after):
            if np.linalg.norm(new - old) > 0.01:
                ax.annotate('', xy=new, xytext=old,
                            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

    for i, c in enumerate(c_show):
        ax.scatter(*c, s=200, color=COLORS[i], edgecolors='black',
                   linewidths=2, marker='X', zorder=5)

    step = 'Update Centroids' if is_update else 'Assign Clusters'
    ax.set_title(f'Iteration {iter_idx + 1} — {step}', fontsize=13, fontweight='bold')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
    ax.set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)

anim = animation.FuncAnimation(fig, animate_frame,
                                frames=len(hist['labels']) * 2,
                                interval=800, repeat=True)
plt.close(fig)
HTML(anim.to_html5_video())

---
## 3. Elbow Method — Choosing K

Run K-Means for K = 1 … 10 and plot WCSS.  
The **"elbow"** is where adding clusters stops helping much.

In [ ]:
K_range = range(1, 11)
wcss_values = [kmeans(X, k)[2] for k in K_range]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), wcss_values, 'o-', color='#2c3e50', linewidth=2, markersize=8)
ax.axvline(x=3, color='#e74c3c', linestyle='--', alpha=0.7, label='Elbow at K=3')
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Within-Cluster Sum of Squares', fontsize=12)
ax.set_title('Elbow Method', fontsize=14, fontweight='bold')
ax.set_xticks(list(K_range))
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('👆 The sharp bend at K=3 tells us 3 clusters is optimal.')

---
## 4. The sklearn Way

Side-by-side comparison + elbow method in just a few lines.

In [ ]:
from sklearn.cluster import KMeans

sk = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)

# --- Side-by-side clustering ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, lbl, ctrs, title in [
    (axes[0], labels, centroids, 'Our Implementation'),
    (axes[1], sk.labels_, sk.cluster_centers_, 'sklearn KMeans'),
]:
    for i in range(K):
        ax.scatter(X[lbl == i, 0], X[lbl == i, 1], s=20, alpha=0.5, color=COLORS[i])
    for i, c in enumerate(ctrs):
        ax.scatter(*c, s=200, color=COLORS[i], edgecolors='black',
                   linewidths=2, marker='X', zorder=5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
plt.suptitle('From Scratch  vs  scikit-learn', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()
print(f'Our WCSS:     {wcss:.2f}')
print(f'sklearn WCSS: {sk.inertia_:.2f}')

# --- Elbow in 3 lines ---
sk_wcss = [KMeans(k, random_state=42, n_init=10).fit(X).inertia_ for k in range(1, 11)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 11), sk_wcss, 'o-', color='#2c3e50', linewidth=2, markersize=8)
ax.axvline(x=3, color='#e74c3c', linestyle='--', alpha=0.7, label='Elbow at K=3')
ax.set_xlabel('K'); ax.set_ylabel('Inertia (WCSS)')
ax.set_title('Elbow Method (sklearn)', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 11))
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📝 Key Takeaways

| Concept | Detail |
|---|---|
| **Algorithm** | Initialise centroids → assign points → update centroids → repeat |
| **Convergence** | Stops when centroids no longer move |
| **Choosing K** | Elbow method (WCSS), silhouette score, or domain knowledge |
| **Limitation** | Sensitive to initialisation — sklearn uses `n_init=10` to mitigate |
| **Complexity** | O(n · k · d · iterations) — fast for moderate data |

---
## 5. 🧪 Bonus: Neural K-Means in JAX

Can a simple neural network learn K-Means?  
The idea: a single linear layer + softmax acts as a **soft cluster assignment**,  
and the reconstruction loss pushes the weights to become centroids.

```
assignments = softmax(X @ W / τ)     # soft cluster picks
reconstruction = assignments @ W.T   # map back through centroids
loss = ||X - reconstruction||²       # push centroids toward data
```

Lower temperature `τ` → sharper assignments → closer to real K-Means.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit

# --- Model ---
def soft_kmeans_loss(W, X, tau):
    """Forward pass: assign, reconstruct, compute MSE."""
    logits = X @ W / tau                          # (n, k)
    assignments = jax.nn.softmax(logits, axis=1)  # (n, k)
    reconstruction = assignments @ W.T            # (n, d)
    return jnp.mean((X - reconstruction) ** 2)


# --- Training loop (records snapshots for animation) ---
def train_neural_kmeans(X_np, k=3, lr=0.05, steps=300,
                        tau_start=2.0, tau_end=0.1, snapshot_every=1):
    X_jax = jnp.array(X_np)

    # Initialise W randomly — shape (d, k)
    key = jax.random.PRNGKey(0)
    W = jax.random.normal(key, (X_np.shape[1], k)) * 0.5

    grad_fn = jit(grad(soft_kmeans_loss))
    loss_fn = jit(soft_kmeans_loss)

    history = []

    for step in range(steps):
        # Anneal temperature: linear decay
        tau = tau_start + (tau_end - tau_start) * step / steps

        # Gradient step
        g = grad_fn(W, X_jax, tau)
        W = W - lr * g

        # Record snapshot
        if step % snapshot_every == 0 or step == steps - 1:
            logits = X_jax @ W / tau
            assignments = jax.nn.softmax(logits, axis=1)
            hard_labels = jnp.argmax(assignments, axis=1)
            centroids = W.T  # (k, d)
            loss = float(loss_fn(W, X_jax, tau))
            history.append({
                'step': step,
                'labels': np.array(hard_labels),
                'centroids': np.array(centroids),
                'loss': loss,
                'tau': float(tau),
            })

    return history


nn_history = train_neural_kmeans(X, k=K, snapshot_every=10)
print(f'Recorded {len(nn_history)} snapshots')
print(f'Final loss: {nn_history[-1]["loss"]:.4f}  |  Final τ: {nn_history[-1]["tau"]:.3f}')

In [ ]:
# --- Animate the neural K-Means learning ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

losses = [h['loss'] for h in nn_history]
steps  = [h['step'] for h in nn_history]

def animate_nn(frame_idx):
    h = nn_history[frame_idx]

    # Left: cluster assignments
    ax1.clear()
    for i in range(K):
        mask = h['labels'] == i
        ax1.scatter(X[mask, 0], X[mask, 1], s=20, alpha=0.5, color=COLORS[i])
    for i, c in enumerate(h['centroids']):
        ax1.scatter(*c, s=200, color=COLORS[i], edgecolors='black',
                    linewidths=2, marker='X', zorder=5)
    ax1.set_title(f'Step {h["step"]}  |  τ={h["tau"]:.2f}', fontsize=13, fontweight='bold')
    ax1.set_xlabel('x₁'); ax1.set_ylabel('x₂')
    ax1.set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
    ax1.set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)

    # Right: loss curve
    ax2.clear()
    ax2.plot(steps[:frame_idx + 1], losses[:frame_idx + 1],
             '-', color='#2c3e50', linewidth=2)
    ax2.scatter(steps[frame_idx], losses[frame_idx],
                color='#e74c3c', s=80, zorder=5)
    ax2.set_xlabel('Training Step'); ax2.set_ylabel('Loss')
    ax2.set_title('Reconstruction Loss', fontsize=13, fontweight='bold')
    ax2.set_xlim(0, steps[-1])
    ax2.set_ylim(0, max(losses) * 1.1)
    ax2.grid(alpha=0.3)

anim = animation.FuncAnimation(fig, animate_nn,
                                frames=len(nn_history),
                                interval=120, repeat=True)
plt.close(fig)
HTML(anim.to_html5_video())

In [ ]:
# --- Final comparison: Classical vs Neural ---
nn_final = nn_history[-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, lbl, ctrs, title in [
    (axes[0], labels, centroids, 'Classical K-Means'),
    (axes[1], nn_final['labels'], nn_final['centroids'], 'Neural K-Means (JAX)'),
]:
    for i in range(K):
        ax.scatter(X[lbl == i, 0], X[lbl == i, 1], s=20, alpha=0.5, color=COLORS[i])
    for i, c in enumerate(ctrs):
        ax.scatter(*c, s=200, color=COLORS[i], edgecolors='black',
                   linewidths=2, marker='X', zorder=5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
plt.suptitle('Classical  vs  Neural K-Means', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()